# 40 — Temporal backtesting

Rolling-origin temporal backtesting of the fixed ARGOS V1 modeling recipe using **development periods only**. TEST 2025 is excluded by governance.


In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.robustness import load_robustness_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
(paths.root / "reports" / "robustness").mkdir(parents=True, exist_ok=True)
(paths.root / "data" / "m5").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "models" / "m5").mkdir(parents=True, exist_ok=True)

m4_gate = paths.docs / "decisions" / "M4_GATE.md"
assert m4_gate.exists() and "M4 ✅ CLOSED" in m4_gate.read_text(encoding="utf-8"), (
    "M4 no está formalmente cerrado. No iniciar M5."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M5_SPEC = load_robustness_spec(paths.configs / "robustness.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

assert GOLD_SPEC.schema_version == M5_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M5_SPEC.core.benchmark_model_schema
{
    "gold_schema": GOLD_SPEC.schema_version,
    "model_schema": MODEL_SPEC.schema_version,
    "m5_schema": M5_SPEC.schema_version,
    "benchmark_model": M5_SPEC.core.benchmark_model_name,
    "benchmark_calibration": M5_SPEC.core.benchmark_calibration,
    "consumed_test_year": M5_SPEC.governance.consumed_test_year,
    "fresh_oot_year": M5_SPEC.governance.fresh_oot_year,
}

from argos.robustness.backtesting import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm5-40-rolling-origin-backtesting-v1'

In [2]:
from argos.robustness.backtesting import build_temporal_backtesting

result = build_temporal_backtesting(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    robustness_spec=M5_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
    temp_directory=paths.root / ".cache" / "duckdb",
)
result.metrics


,model,split,rows,positives,prevalence,pr_auc,roc_auc,log_loss,brier,ece
0,hist_gradient_boosting_fixed_recipe,backtest_2021,561581,5803,0.010333,0.048415,0.752242,0.223870,0.061779,0.156857
1,hist_gradient_boosting_fixed_recipe,backtest_2022,710534,9289,0.013073,0.072170,0.786582,0.185596,0.047568,0.125963
2,hist_gradient_boosting_fixed_recipe,backtest_2023,843059,15283,0.018128,0.094661,0.813049,0.247852,0.071673,0.160679
3,hist_gradient_boosting_fixed_recipe,backtest_2024,950670,9180,0.009656,0.060340,0.834935,0.232063,0.065063,0.149996


In [3]:
result.ranking


,model,split,review_fraction,reviewed,captured_positives,precision_at_k,recall_at_k,lift_at_k,cumulative_gain
0,hist_gradient_boosting_fixed_recipe,backtest_2021,0.001,562,92,0.163701,0.015854,15.842049,0.015854
1,hist_gradient_boosting_fixed_recipe,backtest_2021,0.005,2808,363,0.129274,0.062554,12.510347,0.062554
2,hist_gradient_boosting_fixed_recipe,backtest_2021,0.010,5616,649,0.115563,0.111839,11.183492,0.111839
3,hist_gradient_boosting_fixed_recipe,backtest_2021,0.025,14040,1090,0.077635,0.187834,7.513101,0.187834
4,hist_gradient_boosting_fixed_recipe,backtest_2021,0.050,28080,1590,0.056624,0.273996,5.479739,0.273996
5,hist_gradient_boosting_fixed_recipe,backtest_2021,0.100,56159,2235,0.039798,0.385146,3.851394,0.385146
6,hist_gradient_boosting_fixed_recipe,backtest_2022,0.001,711,170,0.239100,0.018301,18.289222,0.018301
7,hist_gradient_boosting_fixed_recipe,backtest_2022,0.005,3553,644,0.181255,0.069329,13.864575,0.069329
8,hist_gradient_boosting_fixed_recipe,backtest_2022,0.010,7106,1013,0.142556,0.109054,10.904359,0.109054
9,hist_gradient_boosting_fixed_recipe,backtest_2022,0.025,17764,1957,0.110167,0.210679,8.426864,0.210679


In [4]:
result.folds


,evaluation_year,train_end,train_population_rows,train_population_positives,train_sample_rows,train_sample_positives,evaluation_rows,evaluation_positives,positive_class_weight,iterations_fitted,used_consumed_test_2025
0,2021,2020-12-31,666427,28071,166697,6940,561581,5803,23.019741,120,False
1,2022,2021-12-31,1228008,33874,250000,6818,710534,9289,35.667644,120,False
2,2023,2022-12-31,1938542,43163,250000,5509,843059,15283,44.380287,112,False
3,2024,2023-12-31,2781601,58446,250000,5317,950670,9180,46.018996,120,False


In [5]:
result.validation


,check_id,category,blocking,passed,observed,requirement
0,backtest_fold_count,temporal,True,True,"{""folds"": 4, ""years"": [2021, 2022, 2023, 2024]}",at least 4 chronological development folds are...
1,backtest_never_uses_consumed_test,governance,True,True,"{""consumed_test_year"": 2025, ""evaluation_years...",rolling-origin development backtests never use...
2,backtest_population_sufficient,quality,True,True,"{""min_positives"": 5803, ""min_rows"": 561581}",every backtest fold has sufficient rows and po...
3,backtest_operational_lift,usefulness,True,True,"{""lifts"": [{""lift_at_k"": 11.183492040171773, ""...",every development fold produces lift above ran...


In [6]:
blocking = result.validation.loc[result.validation["blocking"]]
print("40 ✅ PASS" if result.ready_for_next_notebook else "40 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook


40 ✅ PASS
Blocking checks: 4/4
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m5_temporal_backtesting.json
